
# **ML Modelling**

In [2]:
# LOAD DATA
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os, joblib

folders = glob.glob("/content/drive/MyDrive/Int_data_after_corr_*")

if len(folders) == 0:
    raise ValueError("No saved dataset found. Check your folder name.")

latest_folder = max(folders, key=os.path.getmtime)
print("Loading data from:", latest_folder)

# FIX: correct filenames
X_train = joblib.load(os.path.join(latest_folder, "X_train_final.pkl"))
X_test = joblib.load(os.path.join(latest_folder, "X_test_final.pkl"))
y_train = joblib.load(os.path.join(latest_folder, "y_train.pkl"))
y_test = joblib.load(os.path.join(latest_folder, "y_test.pkl"))


# LOAD FEATURE SELECTION RESULTS
fs_folders = glob.glob("/content/drive/MyDrive/Int_FS_results_*")

if len(fs_folders) == 0:
    raise ValueError("No FS results found.")

latest_fs_folder = max(fs_folders, key=os.path.getmtime)
print("Loading FS from:", latest_fs_folder)

fs_methods = {}

for file in os.listdir(latest_fs_folder):
    if file.endswith(".pkl"):
        name = file.replace(".pkl", "")
        fs_methods[name] = joblib.load(os.path.join(latest_fs_folder, file))

# ADD full dataset (no FS)
fs_methods["ALL_FEATURES"] = X_train.columns.tolist()


# SMOTE (TRAIN ONLY)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Import
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# EVALUATION METRICS FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision (W)": precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall (W)": recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "F1 (W)": f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "F1 (Macro)": f1_score(y_test, y_pred, average='macro', zero_division=0) * 100,
    }

def print_results(results):
    for k, v in results.items():
        print(f"{k}: {v:.2f}%")

Mounted at /content/drive
Loading data from: /content/drive/MyDrive/Int_data_after_corr_20260507_1548
Loading FS from: /content/drive/MyDrive/Int_FS_results_20260507_1548


In [ ]:
# Print Loaded Feature Sets

print("\n================ FEATURE SETS LOADED ================")

for method, feats in fs_methods.items():
    print(f"\n{method}:")
    print(f"Number of features: {len(feats)}")
    print("Features:", feats)



================ FEATURE SETS LOADED ================

ANOVA:
Number of features: 15
Features: ['Test Today - peak HR', 'Age', 'RISK  - Risk Type', 'Predicted Risk Level Encoded', 'Test Today - Termination Cause', 'Test Today - METS', 'Exercise Habit - Mode', 'Occupation', 'Risk Factor - Stress', 'Exercise Habit - Duration', 'Risk Factor - ECHO - EF', 'Diagnosis', 'ROM', 'ECG Resting', 'Risk Factor - DM']

Mutual_Info:
Number of features: 15
Features: ['Test Today - peak HR', 'Test Today - METS', 'Functional Activity', 'Age', 'RISK  - Risk Type', 'Living Environment', 'ROM', 'Risk Factor - Smoking', 'Predicted Risk Level Encoded', 'Weekly_Exercise_Duration', 'Walking', 'Gait', 'Risk Factor - Stress', 'Posture', 'Lives With']

RFE_LR:
Number of features: 15
Features: ['Age', 'Test Today - peak HR', 'Predicted Risk Level Encoded', 'RISK  - Risk Type', 'Risk Factor - Stress', 'Gender', 'Risk Factor - DM', 'Test Today - Termination Cause', 'Risk Factor - Family hx', 'Risk Factor - HPL', '

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# **Conventional Models**

*   Decision Tree
*   Logistic Regression (Classifier)
*   k-Nearest Neighbors (k-NN)



In [ ]:
# CONVENTIONAL MODELS

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# MODELS
# =========================================================

models = {

    "Decision Tree": (

        DecisionTreeClassifier(
            class_weight='balanced',
            random_state=SEED
        ),

        {
            "max_depth": list(range(2, 30))
        }
    ),

    "Logistic Regression": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("model", LogisticRegression(
                max_iter=5000,
                random_state=SEED
            ))
        ]),

        {
            "model__C": np.logspace(-3, 2, 10)
        }
    ),

    "k-NN": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("model", KNeighborsClassifier())
        ]),

        {
            "model__n_neighbors": list(range(3, 15))
        }
    )
}

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== CONVENTIONAL | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    for name, (model, params) in models.items():

        print(f"\n{name}")

        total_space = int(
            np.prod([len(v) for v in params.values()])
        )

        n_iter = min(10, total_space)

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=params,

            n_iter=n_iter,

            cv=cv,

            scoring='accuracy',

            n_jobs=1,

            random_state=SEED
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)

        print_results(
            evaluate(y_test, y_pred)
        )

Using 4-Fold Stratified CV


========== CONVENTIONAL | FEATURE SET: ANOVA ==========

Decision Tree
Best Params: {'max_depth': 27}
Accuracy: 55.56%
Precision (W): 55.98%
Recall (W): 55.56%
F1 (W): 55.75%
Precision (Macro): 35.45%
Recall (Macro): 35.49%
F1 (Macro): 35.47%

Logistic Regression
Best Params: {'model__C': np.float64(7.742636826811277)}
Accuracy: 57.14%
Precision (W): 67.47%
Recall (W): 57.14%
F1 (W): 61.38%
Precision (Macro): 37.71%
Recall (Macro): 38.28%
F1 (Macro): 36.00%

k-NN
Best Params: {'model__n_neighbors': 3}
Accuracy: 57.14%
Precision (W): 57.45%
Recall (W): 57.14%
F1 (W): 57.17%
Precision (Macro): 40.67%
Recall (Macro): 37.22%
F1 (Macro): 38.50%


========== CONVENTIONAL | FEATURE SET: Mutual_Info ==========

Decision Tree
Best Params: {'max_depth': 27}
Accuracy: 65.08%
Precision (W): 64.44%
Recall (W): 65.08%
F1 (W): 64.64%
Precision (Macro): 44.38%
Recall (Macro): 40.51%
F1 (Macro): 42.00%

Logistic Regression
Best Params: {'model__C': np.float64(100.0)}
Accura

In [ ]:
# SVM MODEL

import numpy as np
import random
import os

from collections import Counter

from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# Automatically adjusts to minority class
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# SVM PIPELINE
# ============================================

svm_model = ImbPipeline([

    ("scaler", StandardScaler()),

    ("ros", RandomOverSampler(
        random_state=SEED
    )),

    ("svm", SVC(
        kernel='rbf',
        probability=True,
        random_state=SEED
    ))
])

# ============================================
# HYPERPARAMETERS
# ============================================

param_dist = {

    "svm__C": [
        0.1,
        1,
        10
    ],

    "svm__gamma": [
        "scale",
        "auto"
    ]
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== SVM | FEATURE SET: {fs_name} ==========")

    # ----------------------------------------
    # SELECT FEATURES
    # ----------------------------------------

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # ----------------------------------------
    # SAFE n_iter
    # ----------------------------------------

    total_space = int(np.prod([
        len(v) for v in param_dist.values()
    ]))

    n_iter = min(10, total_space)

    # ----------------------------------------
    # RANDOM SEARCH CV
    # ----------------------------------------

    search = RandomizedSearchCV(
        estimator=svm_model,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,
        random_state=SEED
    )

    # ----------------------------------------
    # TRAIN
    # ----------------------------------------

    search.fit(X_train_sel, y_train)

    # ----------------------------------------
    # TEST
    # ----------------------------------------

    y_pred = search.predict(X_test_sel)

    # ----------------------------------------
    # RESULTS
    # ----------------------------------------

    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 4-Fold Stratified CV


========== SVM | FEATURE SET: ANOVA ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 1}
Accuracy: 68.25%
Precision (W): 64.46%
Recall (W): 68.25%
F1 (W): 65.81%
Precision (Macro): 33.95%
Recall (Macro): 34.15%
F1 (Macro): 33.75%


========== SVM | FEATURE SET: Mutual_Info ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 1}
Accuracy: 57.14%
Precision (W): 62.32%
Recall (W): 57.14%
F1 (W): 59.22%
Precision (Macro): 39.43%
Recall (Macro): 43.83%
F1 (Macro): 40.08%


========== SVM | FEATURE SET: RFE_LR ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 10}
Accuracy: 61.90%
Precision (W): 57.79%
Recall (W): 61.90%
F1 (W): 59.52%
Precision (Macro): 29.55%
Recall (Macro): 30.45%
F1 (Macro): 29.83%


========== SVM | FEATURE SET: RFE_SVM ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 1}
Accuracy: 71.43%
Precision (W): 70.38%
Recall (W): 71.43%
F1 (W): 70.47%
Precision (Macro): 37.72%
Recall (Macro): 36.53%
F1 (Macro): 36.86%


=

# **Ensemble models**

*   LightGBM
*   XGBoost
*   CatBoost


In [ ]:
# ENSEMBLE MODELS

import numpy as np
import random
import os

from collections import Counter

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# INSTALL MISSING PACKAGES
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# CatBoost
try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError:
    install("catboost")
    from catboost import CatBoostClassifier

# LightGBM
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError:
    install("lightgbm")
    from lightgbm import LGBMClassifier

# XGBoost
try:
    import xgboost as xgb
except ModuleNotFoundError:
    install("xgboost")
    import xgboost as xgb

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# MODELS
# ============================================

ensembles = {

    "LightGBM": (
        LGBMClassifier(
            class_weight='balanced',
            random_state=SEED,
            verbosity=-1,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
        }
    ),

    "XGBoost": (
        xgb.XGBClassifier(
            eval_metric='mlogloss',
            random_state=SEED,
            use_label_encoder=False,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
        }
    ),

    "CatBoost": (
        CatBoostClassifier(
            auto_class_weights="Balanced",
            verbose=0,
            random_seed=SEED
        ),
        {
            "iterations": [200, 400],
            "depth": [4, 6],
            "learning_rate": [0.05, 0.1],
        }
    ),
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== ENSEMBLE | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    for name, (model, params) in ensembles.items():

        print(f"\n{name}")

        total_space = int(np.prod([len(v) for v in params.values()]))
        n_iter = min(10, total_space)

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=n_iter,
            cv=cv,
            scoring='accuracy',
            n_jobs=1,
            random_state=SEED
        )

        search.fit(X_train_sel, y_train)

        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)
        print_results(evaluate(y_test, y_pred))

Using 4-Fold Stratified CV


========== ENSEMBLE | FEATURE SET: ANOVA ==========

Random Forest
Best Params: {'n_estimators': 200, 'max_depth': 10}
Accuracy: 63.49%
Precision (W): 58.47%
Recall (W): 63.49%
F1 (W): 58.87%
Precision (Macro): 30.76%
Recall (Macro): 30.04%
F1 (Macro): 29.15%

LightGBM
Best Params: {'n_estimators': 100, 'learning_rate': 0.05}
Accuracy: 68.25%
Precision (W): 66.61%
Recall (W): 68.25%
F1 (W): 67.32%
Precision (Macro): 49.43%
Recall (Macro): 49.50%
F1 (Macro): 49.40%

XGBoost
Best Params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Accuracy: 63.49%
Precision (W): 58.30%
Recall (W): 63.49%
F1 (W): 60.31%
Precision (Macro): 30.08%
Recall (Macro): 31.11%
F1 (Macro): 30.29%

CatBoost
Best Params: {'learning_rate': 0.05, 'iterations': 400, 'depth': 4}
Accuracy: 61.90%
Precision (W): 62.55%
Recall (W): 61.90%
F1 (W): 60.93%
Precision (Macro): 54.55%
Recall (Macro): 37.59%
F1 (Macro): 41.60%


========== ENSEMBLE | FEATURE SET: Mutual_Info ==========


# **Deep learning**

*   FCN
*   BPNN
*   Autoencoder Classifier






In [ ]:
# DEEP LEARNING MODELS

import numpy as np
import tensorflow as tf
import random
import os

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# FIXED SEED
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

os.environ['TF_DETERMINISTIC_OPS'] = '1'

# SETTINGS
EPOCHS = 40
BATCH_SIZE = 32

# CLASS WEIGHTS
def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

# MODEL DEFINITIONS
def build_fcn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_bpnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# AUTOENCODER + CLASSIFIER
def build_autoencoder_classifier(input_dim, num_classes):

    inputs = tf.keras.Input(shape=(input_dim,))

    encoded = tf.keras.layers.Dense(64, activation='relu')(inputs)
    encoded = tf.keras.layers.Dense(32, activation='relu')(encoded)

    decoded = tf.keras.layers.Dense(64, activation='relu')(encoded)
    decoded = tf.keras.layers.Dense(input_dim, activation='linear', name="reconstruction")(decoded)

    classifier = tf.keras.layers.Dense(num_classes, activation='softmax', name="classification")(encoded)

    model = tf.keras.Model(inputs=inputs, outputs=[decoded, classifier])

    return model

# TRAINING FUNCTIONS
def train_standard(model_fn, X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = model_fn(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    class_weights = get_class_weights(y_train)

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        class_weight=class_weights,
        shuffle=True
    )

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    return evaluate(y_test, y_pred)

def train_autoencoder(X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = build_autoencoder_classifier(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss={
            "reconstruction": "mse",
            "classification": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "reconstruction": 0.3,
            "classification": 1.0
        },
        metrics={"classification": "accuracy"}
    )

    model.fit(
        X_train,
        {"reconstruction": X_train, "classification": y_train},
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        shuffle=True
    )

    preds = model.predict(X_test, verbose=0)[1]
    y_pred = np.argmax(preds, axis=1)

    return evaluate(y_test, y_pred)

# MAIN LOOP
for fs_name, features in fs_methods.items():

    print(f"\n\n========== DEEP LEARNING | FEATURE SET: {fs_name} ==========")

    # SCALE
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train[features])
    X_test_s = scaler.transform(X_test[features])

    # FCN
    print("\nFCN")
    print_results(train_standard(build_fcn, X_train_s, y_train, X_test_s, y_test))

    # BPNN
    print("\nBPNN")
    print_results(train_standard(build_bpnn, X_train_s, y_train, X_test_s, y_test))

    # AUTOENCODER
    print("\nAutoencoder Classifier")
    print_results(train_autoencoder(X_train_s, y_train, X_test_s, y_test))



========== DEEP LEARNING | FEATURE SET: ANOVA ==========

FCN
Accuracy: 60.32%
Precision (W): 59.05%
Recall (W): 60.32%
F1 (W): 59.64%
Precision (Macro): 30.63%
Recall (Macro): 30.86%
F1 (Macro): 30.72%

BPNN
Accuracy: 58.73%
Precision (W): 55.59%
Recall (W): 58.73%
F1 (W): 57.10%
Precision (Macro): 28.37%
Recall (Macro): 29.67%
F1 (Macro): 28.99%

CNN


Accuracy: 65.08%
Precision (W): 61.38%
Recall (W): 65.08%
F1 (W): 63.01%
Precision (Macro): 31.92%
Recall (Macro): 32.83%
F1 (Macro): 32.27%

Autoencoder Classifier
Accuracy: 60.32%
Precision (W): 55.02%
Recall (W): 60.32%
F1 (W): 57.12%
Precision (Macro): 28.07%
Recall (Macro): 29.26%
F1 (Macro): 28.38%


========== DEEP LEARNING | FEATURE SET: Mutual_Info ==========

FCN
Accuracy: 63.49%
Precision (W): 66.81%
Recall (W): 63.49%
F1 (W): 64.90%
Precision (Macro): 38.36%
Recall (Macro): 39.85%
F1 (Macro): 38.32%

BPNN
Accuracy: 66.67%
Precision (W): 69.28%
Recall (W): 66.67%
F1 (W): 67.84%
Precision (Macro): 40.38%
Recall (Macro): 41.70%
F1 (Macro): 40.58%

CNN
Accuracy: 66.67%
Precision (W): 69.52%
Recall (W): 66.67%
F1 (W): 68.01%
Precision (Macro): 41.63%
Recall (Macro): 41.70%
F1 (Macro): 41.48%

Autoencoder Classifier
Accuracy: 74.60%
Precision (W): 70.51%
Recall (W): 74.60%
F1 (W): 70.87%
Precision (Macro): 38.01%
Recall (Macro): 36.78%
F1 (Macro): 36.40%


========== DEEP LEARNIN

# **Transformer models**

*   FT-Transformer
*   TabTransformer
*   TabNet



In [ ]:
# TRANSFORMER MODELS

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.ensemble import HistGradientBoostingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# MODELS
# =========================================================

transformer_models = {

    # FT-Transformer Approximation
    "FT-Transformer (Approx)": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("mlp", MLPClassifier(
                hidden_layer_sizes=(256, 128, 64),
                max_iter=1500,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=SEED
            ))
        ]),

        {
            "mlp__alpha": [0.0001, 0.001]
        }
    ),

    # TabTransformer Approximation
    "TabTransformer (Approx)": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("mlp", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                max_iter=1200,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=SEED
            ))
        ]),

        {
            "mlp__alpha": [0.0001, 0.001]
        }
    ),

}

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TRANSFORMER | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    for name, (model, params) in transformer_models.items():

        print(f"\n{name}")

        total_space = int(
            np.prod([len(v) for v in params.values()])
        )

        n_iter = min(10, total_space)

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=params,

            n_iter=n_iter,

            cv=cv,

            scoring='accuracy',

            n_jobs=1,

            random_state=SEED
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)

        print_results(
            evaluate(y_test, y_pred)
        )

Using 4-Fold Stratified CV


========== TRANSFORMER | FEATURE SET: ANOVA ==========

FT-Transformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 63.49%
Precision (W): 60.95%
Recall (W): 63.49%
F1 (W): 62.06%
Precision (Macro): 31.74%
Recall (Macro): 32.17%
F1 (Macro): 31.87%

TabTransformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 52.38%
Precision (W): 57.86%
Recall (W): 52.38%
F1 (W): 54.91%
Precision (Macro): 29.90%
Recall (Macro): 26.50%
F1 (Macro): 28.06%

Attention-like Boosting (HistGB)
Best Params: {'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 68.25%
Precision (W): 67.54%
Recall (W): 68.25%
F1 (W): 66.88%
Precision (Macro): 57.50%
Recall (Macro): 41.82%
F1 (Macro): 45.39%


========== TRANSFORMER | FEATURE SET: Mutual_Info ==========

FT-Transformer (Approx)
Best Params: {'mlp__alpha': 0.001}
Accuracy: 65.08%
Precision (W): 66.16%
Recall (W): 65.08%
F1 (W): 65.19%
Precision (Macro): 40.17%
Recall (Macro): 39.97%
F1 (Macro): 39.64%

TabTransformer (App

In [4]:
# TABNET MODEL

import numpy as np
import pandas as pd
import random
import os
import warnings

from collections import Counter

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

# ============================================
# SUPPRESS TABNET WARNING
# ============================================

warnings.filterwarnings(
    "ignore",
    message="No early stopping will be performed"
)

# ============================================
# INSTALL TABNET IF MISSING
# ============================================

import sys
import subprocess

def install(package):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", package]
    )

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ModuleNotFoundError:
    install("pytorch-tabnet")
    from pytorch_tabnet.tab_model import TabNetClassifier

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# TABNET WRAPPER
# ============================================

class TabNetWrapper(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.3,
        lambda_sparse=1e-4
    ):

        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse

    def fit(self, X, y):

        # Convert dataframe to numpy
        X_np = X.values if hasattr(X, "values") else X

        # Internal validation split
        X_train_inner, X_valid_inner, y_train_inner, y_valid_inner = train_test_split(
            X_np,
            y,
            test_size=0.1,
            stratify=y,
            random_state=SEED
        )

        # Create model
        self.model_ = TabNetClassifier(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            seed=SEED,
            verbose=0
        )

        # Train with early stopping
        self.model_.fit(
            X_train=X_train_inner,
            y_train=y_train_inner,

            eval_set=[
                (X_valid_inner, y_valid_inner)
            ],

            eval_metric=["accuracy"],

            max_epochs=100,
            patience=20,

            batch_size=32,
            virtual_batch_size=16
        )

        return self

    def predict(self, X):

        X_np = X.values if hasattr(X, "values") else X

        return self.model_.predict(X_np)

    def predict_proba(self, X):

        X_np = X.values if hasattr(X, "values") else X

        return self.model_.predict_proba(X_np)

# ============================================
# PIPELINE
# ============================================

tabnet_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", TabNetWrapper())
])

# ============================================
# PARAMETER SPACE
# ============================================

param_dist = {

    "model__n_d": [8, 16, 32],

    "model__n_a": [8, 16, 32],

    "model__n_steps": [3, 5],

    "model__gamma": [1.0, 1.3],

    "model__lambda_sparse": [1e-3, 1e-4]
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TABNET | FEATURE SET: {fs_name} ==========")

    # Feature selection
    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # Prevent n_iter warning
    total_space = int(
        np.prod([len(v) for v in param_dist.values()])
    )

    n_iter = min(10, total_space)

    # Randomized Search
    search = RandomizedSearchCV(
        estimator=tabnet_pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,
        random_state=SEED
    )

    # TRAIN
    search.fit(X_train_sel, y_train)

    # TEST
    y_pred = search.predict(X_test_sel)

    # RESULTS
    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 4-Fold Stratified CV


========== TABNET | FEATURE SET: ANOVA ==========

Early stopping occurred at epoch 49 with best_epoch = 29 and best_val_0_accuracy = 0.89474


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 49 with best_epoch = 29 and best_val_0_accuracy = 0.84211


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 21 with best_epoch = 1 and best_val_0_accuracy = 0.78947


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 33 with best_epoch = 13 and best_val_0_accuracy = 0.73684


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 25 with best_epoch = 5 and best_val_0_accuracy = 0.76
Best Params: {'model__n_steps': 3, 'model__n_d': 32, 'model__n_a': 8, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 60.32%
Precision (W): 54.18%
Recall (W): 60.32%
F1 (W): 55.15%
Precision (Macro): 27.54%
Recall (Macro): 27.66%
F1 (Macro): 26.35%


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
